In [ ]:
REPO_ROOT = "../"

%matplotlib inline
import os
os.chdir(os.path.dirname(REPO_ROOT))
%load_ext autoreload
import torch
import torch.nn as nn
import pytorch_lightning as pl
import albumentations as A
import cv2
import matplotlib.pyplot as plt
from data.retrieval_module_fixed_split import RetrievalDataModuleFixedSplit
from data.dinov2_adaptations import AverageFirstChannelToThird
from retrieval.knn_seg_hbird import KNNHummingBirdSegmentation

# Set random seeds
seed = 42
pl.seed_everything(seed, workers=True)

# Enhanced thesis-ready configuration
plt.rcParams['font.size'] = 14          # Slightly larger base font
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 18     # More prominent titles
plt.rcParams['axes.labelsize'] = 14     # Clearer axis labels
plt.rcParams['xtick.labelsize'] = 12    # Readable tick labels
plt.rcParams['ytick.labelsize'] = 12    # Readable tick labels
plt.rcParams['legend.fontsize'] = 12    # Add legend font size
plt.rcParams['figure.titlesize'] = 20   # Overall figure title
plt.rcParams['lines.linewidth'] = 2     # Thicker lines for clarity
plt.rcParams['axes.linewidth'] = 1.2    # Thicker axes
plt.rcParams['grid.alpha'] = 0.3        # Subtle grid if used
plt.rcParams['savefig.dpi'] = 300       # High resolution for print
plt.rcParams['savefig.bbox'] = 'tight'  # Remove extra whitespace
plt.rcParams['figure.figsize'] = (8, 6) # Good default size

transform = A.Compose(
                [
                    A.PadIfNeeded(
                        min_height=532,
                        min_width=532,
                        # Avoids reflective padding
                        border_mode=cv2.BORDER_CONSTANT,
                        value=(0, 0, 0),
                        p=1,
                    ),
                    A.CenterCrop(
                        532,
                        532,
                    ),
                    AverageFirstChannelToThird(p=1.0),  # Convert 2-channel to 3-channel
                ]
            )

data_module = RetrievalDataModuleFixedSplit(
    data_path="./datasets/pretrain_split/",
    batch_size=16,
    num_workers=4,
    transform=transform,
    eval_transform=transform,
    train_dir="train",
    holdout_dir="test",
    classes=["wire", "ball", "wedge", "epoxy"],
    input_resolution=(532, 532),
)

data_module.setup("test")
train_loader = data_module.train_dataloader()
test_loader = data_module.test_dataloader()

encoder = torch.hub.load(
            'facebookresearch/dinov2', 
            'dinov2_vits14'
        )

encoder = encoder.to("cuda")

class dinoV2Wrapper(nn.Module):
    def __init__(self, encoder):
        super(dinoV2Wrapper, self).__init__()
        self.encoder = encoder
        self.patch_embed = encoder.patch_embed  # This is the patch embedding layer
        self.patch_size = 14
    def forward(self, x):
        output = self.encoder.forward_features(x)
        return output

encoder = dinoV2Wrapper(encoder)


knn_evaluator_test = KNNHummingBirdSegmentation(
    encoder=encoder,
    train_loader=train_loader,
    val_loader=test_loader,
    batch_size=16,
    profile_time=True,
    classes=["wire", "ball", "wedge", "epoxy"],
    average_patches=False,
    profile_memory=True,
)

print(data_module.holdout_dir)

<REPO_ROOT>/segmentation/models/vit/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
<REPO_ROOT>/segmentation/models/vit/layers/attention.py:23: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
<HOME>/miniconda3/envs/thesis/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
<REPO_ROOT>/segmentation/models/vit/layers/block.py:30: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")
<HOME>/miniconda3/envs/thesis/lib/python3.10/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, pl

Using provided transform for training.


Using cache found in <HOME>/.cache/torch/hub/facebookresearch_dinov2_main
<HOME>/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
<HOME>/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
<HOME>/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


Dataset size: 500


<REPO_ROOT>/segmentation/models/vit/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
<REPO_ROOT>/segmentation/models/vit/layers/attention.py:23: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
<REPO_ROOT>/segmentation/models/vit/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
<REPO_ROOT>/segmentation/models/vit/layers/attention.py:23: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
<REPO_ROOT>/segmentation/models/vit/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
<REPO_ROOT>/segmentation/models/vit/layers/attention.py:23: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
<REPO_ROOT>/segmentation/models/vit/layers/swiglu_ffn.p

Time taken to access data: 32.27 seconds
Memory usage cuda: 84.14 MB
Time taken to access data: 0.00 seconds
Memory usage cuda: 326.67 MB
Time taken to access data: 0.00 seconds
Memory usage cuda: 327.10 MB
Time taken to access data: 0.00 seconds
Memory usage cuda: 327.19 MB
Time taken to access data: 8.57 seconds
Memory usage cuda: 326.59 MB
Time taken to access data: 0.00 seconds
Memory usage cuda: 326.67 MB
Time taken to access data: 0.00 seconds
Memory usage cuda: 327.10 MB
Time taken to access data: 0.00 seconds
Memory usage cuda: 326.68 MB
Time taken to access data: 8.67 seconds
Memory usage cuda: 326.58 MB
Time taken to access data: 6.42 seconds
Memory usage cuda: 326.67 MB
Time taken to access data: 0.00 seconds
Memory usage cuda: 326.59 MB
Time taken to access data: 0.00 seconds
Memory usage cuda: 327.19 MB
Time taken to access data: 5.65 seconds
Memory usage cuda: 326.58 MB
Time taken to access data: 0.00 seconds
Memory usage cuda: 326.76 MB
Time taken to access data: 0.00 se

## Distance-based Patch Level Retrieval

### Evaluating Test Set

In [ ]:
# retrieval_module = RetrievalDataModule(
#     data_path="./datasets/pretrain_split/",
#     batch_size=32,
#     classes=config.classes,
#     num_workers=4,
#     input_resolution=config.input_resolution,
#     train_size=1.0,
# )

# retrieval_module.setup("train")
# train_loader = retrieval_module.train_dataloader()
# retrieval_module.setup("test")
# test_loader = retrieval_module.test_dataloader()
# len(test_loader.dataset)

# with open(f"{REPO_ROOT}/data/retrieval_module.py", "r") as f:
#     t = f.read()
#     print(t)

# print(retrieval_module.holdout_img_list)

# for b in test_loader:
#     print(b['image'].shape)
#     break

# knn_evaluator_test = KNNHummingBirdSegmentation(
#     encoder=encoder,
#     train_loader=train_loader,
#     val_loader=test_loader,
#     classes=config.classes,
#     batch_size=32,
#     average_patches=False,
# )

# Summary (sorted by best mean IoU):
#     class  mean_miou  k  threshold
#  iou_ball   0.527611  1        0.5
#  iou_wire   0.482797 10        0.2
# iou_epoxy   0.459147  4        0.5
# iou_wedge   0.301099  4        0.5


iou_ball = knn_evaluator_test.evaluate_distance(
    threshold=0.5,
    k=1,
 )["iou_ball"]

print(f"iou_ball: {iou_ball}")

iou_wire = knn_evaluator_test.evaluate_distance(
    threshold=0.2,
    k=10,
 )["iou_wire"]

print(f"iou_wire: {iou_wire}")

iou_epoxy = knn_evaluator_test.evaluate_distance(
    threshold=0.5,
    k=4,
 )["iou_epoxy"]

print(f"iou_epoxy: {iou_epoxy}")

iou_wedge = knn_evaluator_test.evaluate_distance(
    threshold=0.5,
    k=4,
 )["iou_wedge"]

print(f"iou_wedge: {iou_wedge}")

k is: 1
Finding nearest neighbors for 11552 queries with k=1 using faiss method.
Average similarities: 0.9580773115158081
Classwise IoUs: {'iou_wire': 0.7835056185722351, 'iou_ball': 0.7426730394363403, 'iou_wedge': 0.7642857432365417, 'iou_epoxy': 0.8239446878433228}
Time taken to compute classwise IoUs: 0.84 seconds
k is: 1
Finding nearest neighbors for 11552 queries with k=1 using faiss method.
Average similarities: 0.9109921455383301
Classwise IoUs: {'iou_wire': 0.600364089012146, 'iou_ball': 0.9588410258293152, 'iou_wedge': 0.6522563099861145, 'iou_epoxy': 0.15905126929283142}
Time taken to compute classwise IoUs: 0.93 seconds
k is: 1
Finding nearest neighbors for 11552 queries with k=1 using faiss method.
Average similarities: 0.9485735893249512
Skipping class epoxy due to no positive samples in target.
Classwise IoUs: {'iou_wire': 0.39649730920791626, 'iou_ball': 0.9406532049179077, 'iou_wedge': 0.37445345520973206, 'iou_epoxy': None}
Time taken to compute classwise IoUs: 0.53 s

## Attention-based retrieval

In [3]:
# Summary (sorted by best mean IoU):
# class  mean_miou  k  threshold  beta
#  ball   0.514158 50        0.2  0.02
#  wire   0.501113 50        0.2  0.02
# epoxy   0.475544 20        0.4  0.02
# wedge   0.298333 20        0.4  0.02



iou_ball = knn_evaluator_test.evaluate(
    thresholds=[0.2]*4,
    k=50,
    betas=0.02,
)["ball"]


iou_wire = knn_evaluator_test.evaluate(
    thresholds=[0.2]*4,
    k=50,
    betas=0.02,
)["wire"]


iou_epoxy = knn_evaluator_test.evaluate(
    thresholds=[0.4]*4,
    k=20,
    betas=0.02,
)["epoxy"]


iou_wedge = knn_evaluator_test.evaluate(
    thresholds=[0.4]*4,
    k=20,
    betas=0.02,
)["wedge"]

print("Results from KNNHummingBirdSegmentation test:")
print(f"iou_ball: {iou_ball}")
print(f"iou_wire: {iou_wire}")
print(f"iou_epoxy: {iou_epoxy}")
print(f"iou_wedge: {iou_wedge}")

patch_size: 14
input_resolution: (512, 512)
eval_spatial_resolution: 36
Finding nearest neighbors for 11552 queries with k=50 using faiss method.
Resized label hats shape: torch.Size([8, 4, 532, 532])
match: (array([0, 1, 2, 3]), array([0, 1, 2, 3]))
True Positives: [40199, 25396, 521, 48956], False Positives: [28423, 12787, 279, 25049], False Negatives: [1334, 1299, 76, 3958]
Mean IoUs:  [0.5746326262221968, 0.6432298262499366, 0.5947488584474886, 0.6279388940907866]
patch_size: 14
input_resolution: (512, 512)
eval_spatial_resolution: 36
Finding nearest neighbors for 11552 queries with k=50 using faiss method.
Resized label hats shape: torch.Size([8, 4, 532, 532])
match: (array([0, 1, 2, 3]), array([0, 1, 2, 3]))
True Positives: [64333, 17288, 4407, 24030], False Positives: [35481, 2798, 1864, 59978], False Negatives: [3304, 348, 602, 34508]
Mean IoUs:  [0.6238774995636067, 0.8460409122051483, 0.6412047140986469, 0.20275743359546392]
patch_size: 14
input_resolution: (512, 512)
eval_sp